In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import gc

We conduct some early concatenation, transformation, and merging to prepare the datasets for further analysis

<h3>Concat Raw Datasets</h3>

- concat yearly facility, emissions, and EIA data
- for emissions data, we additionally separate datasets by unit's operating and non-operating hours (we define this by whether any emissions are recorded -> if NOx, CO2, and SO2 are null, it is non-operating)


In [36]:
# Concat all facility csv files into a single df

facility_folder = Path('Facility')

df_list = []
for file in facility_folder.iterdir():
    if file.is_file() and file.name.endswith('.csv'):
        df = pd.read_csv(file)
        df_list.append(df)

campd_facility = pd.concat(df_list, ignore_index=True)

output_path = Path('Facility/Concatted File/facility_2019_2024.csv')
campd_facility.to_csv(output_path, index=False)

In [ ]:
# Concat all ny csv files into a single df

ny_folder = Path('NY/Hourly')

df_list = []
for file in ny_folder.iterdir():
    if file.is_file() and file.name.endswith('.csv'):
        df = pd.read_csv(file)
        df_list.append(df)

campd_ny = pd.concat(df_list, ignore_index=True)

# save full operating and non-operating dfs separately

filtered_out_rows_campd_ny = campd_ny[campd_ny["SO2 Mass (lbs)"].isna() & campd_ny["CO2 Mass (short tons)"].isna() & campd_ny["NOx Mass (lbs)"].isna()]
filtered_out_rows_campd_ny.to_csv('NY/Hourly/Concatted File/ny_2019_2024_NonOperating.csv', index=False)

campd_ny = campd_ny[~(campd_ny["SO2 Mass (lbs)"].isna()) & ~(campd_ny["CO2 Mass (short tons)"].isna()) & ~(campd_ny["NOx Mass (lbs)"].isna())]
campd_ny.to_csv('NY/Hourly/Concatted File/ny_2019_2024_Operating.csv', index=False)

del filtered_out_rows_campd_ny, campd_ny
gc.collect()

In [ ]:
# Concat all wy csv files into a single df

wy_folder = Path('WY/Hourly')

df_list = []
for file in wy_folder.iterdir():
    if file.is_file() and file.name.endswith('.csv'):
        df = pd.read_csv(file)
        df_list.append(df)

campd_wy = pd.concat(df_list, ignore_index=True)

# save full operating and non-operating dfs separately

filtered_out_rows_campd_wy = campd_wy[campd_wy["SO2 Mass (lbs)"].isna() & campd_wy["CO2 Mass (short tons)"].isna() & campd_wy["NOx Mass (lbs)"].isna()]
filtered_out_rows_campd_wy.to_csv('WY/Hourly/Concatted File/wy_2019_2024_NonOperating.csv', index=False)

campd_wy = campd_wy[~(campd_wy["SO2 Mass (lbs)"].isna()) & ~(campd_wy["CO2 Mass (short tons)"].isna()) & ~(campd_wy["NOx Mass (lbs)"].isna())]
campd_wy.to_csv('WY/Hourly/Concatted File/wy_2019_2024_Operating.csv', index=False)

del filtered_out_rows_campd_wy, campd_wy
gc.collect()

In [ ]:
# Concat all tx csv files into a single df

tx_folder = Path('TX/Hourly')

df_list = []
for file in tx_folder.iterdir():
    if file.is_file() and file.name.endswith('.csv'):
        df = pd.read_csv(file)
        df_list.append(df)

campd_tx = pd.concat(df_list, ignore_index=True)

# save full operating and non-operating dfs separately

filtered_out_rows_campd_tx = campd_tx[campd_tx["SO2 Mass (lbs)"].isna() & campd_tx["CO2 Mass (short tons)"].isna() & campd_tx["NOx Mass (lbs)"].isna()]
filtered_out_rows_campd_tx.to_csv('TX/Hourly/Concatted File/tx_2019_2024_NonOperating.csv', index=False)

campd_tx = campd_tx[~(campd_tx["SO2 Mass (lbs)"].isna()) & ~(campd_tx["CO2 Mass (short tons)"].isna()) & ~(campd_tx["NOx Mass (lbs)"].isna())]
campd_tx.to_csv('TX/Hourly/Concatted File/tx_2019_2024_Operating.csv', index=False)

del filtered_out_rows_campd_tx, campd_tx
gc.collect()

In [ ]:
# get all EIA data

eia_folder = Path('EIA')

# since the EIA data is in xlsx format with multiple sheets and does not have a year column
# we use pd.read_excel to extract the relevant sheet and add the year from the filename as a column

df_list = []
for file in eia_folder.iterdir():
    if file.is_file() and file.name.endswith('.xlsx'):
        # extract the year from filename and add as column
        match = re.search(r'_Y(\d{4})\.xlsx', str(file))
        if match:
            year = int(match.group(1))
            df = pd.read_excel(file, sheet_name="Operable", header=1)
            df["Year"] = year
            df_list.append(df)

eia_generator = pd.concat(df_list, ignore_index=True)

# filter for rows in campd_ny, campd_wy, and campd_tx
eia_generator = eia_generator[eia_generator["Plant Code"].isin(campd_ny["Facility ID"]) | 
                                eia_generator["Plant Code"].isin(campd_wy["Facility ID"]) | 
                                eia_generator["Plant Code"].isin(campd_tx["Facility ID"])]

output_path = Path('EIA/Concatted File/eia_generator_2019_2023.csv')
eia_generator.to_csv(output_path, index=False)

<h3>Transform Emission Data</h3>

- read in the (operating) emissions datasets, adjust their datatypes, and print their shapes

In [2]:
# define data type dictionary for campd dataframes
campd_type_dict = {
    "State": "string",
    "Facility Name": "string",
    "Facility ID": "Int64",
    "Unit ID": "string",
    "Associated Stacks": "string",
    "Gross Load (MW)": "float64",
    "Steam Load (1000 lb/hr)": "float64",
    "SO2 Mass (lbs)": "float64",
    "SO2 Mass Measure Indicator": "string",
    "SO2 Rate (lbs/mmBtu)": "float64",
    "SO2 Rate Measure Indicator": "string",
    "CO2 Mass (short tons)": "float64",
    "CO2 Mass Measure Indicator": "string",
    "CO2 Rate (short tons/mmBtu)": "float64",
    "CO2 Rate Measure Indicator": "string",
    "NOx Mass (lbs)": "float64",
    "NOx Mass Measure Indicator": "string",
    "NOx Rate (lbs/mmBtu)": "float64",
    "NOx Rate Measure Indicator": "string",
    "Heat Input (mmBtu)": "float64",
    "Heat Input Measure Indicator": "string",
    "Primary Fuel Type": "string",
    "Secondary Fuel Type": "string",
    "Unit Type": "string",
    "SO2 Controls": "string",
    "NOx Controls": "string",
    "PM Controls": "string",
    "Hg Controls": "string",
    "Program Code": "string",
    "Hour": "Int64",
    "Operating Time": "float64"
}

<h4>Wyoming</h4>

In [3]:
campd_wy = pd.read_csv('WY/Hourly/Concatted File/wy_2019_2024_Operating.csv', dtype=campd_type_dict, parse_dates=["Date"])

In [4]:
campd_wy.shape

(993428, 32)

<h4>New York</h4>

In [5]:
campd_ny = pd.read_csv('NY/Hourly/Concatted File/ny_2019_2024_Operating.csv', dtype=campd_type_dict, parse_dates=["Date"])

In [6]:
campd_ny.shape

(2400701, 32)

<h4>Texas</h4>

In [7]:
campd_tx = pd.read_csv('TX/Hourly/Concatted File/tx_2019_2024_Operating.csv', dtype=campd_type_dict, parse_dates=["Date"])

In [8]:
campd_tx.shape

(9262325, 32)

<h3>Transform Facility Data</h3>

- read in facility data, assign data types, restrict columns, filter relevant rows
- also extract the nameplate capacity per unit (raw data has a string column with the unit's (boiler) associated generators and their respective capacities. we extract these capacities and sum over them for each unit.)

In [ ]:
# read facility data and assign data types

campd_facility = pd.read_csv('Facility/Concatted File/facility_2014_2024.csv', dtype={
    'Year': 'Int64',
    'Facility ID': 'Int64',
    'Unit ID': 'string',
    "State": "string",
    'EPA Region': 'Int64',
    "Source Category": 'string',
    'Latitude': 'float64',
    'Longitude': 'float64',
    'SO2 Phase': 'string',
    'NOx Phase': 'string',
    'Commercial Operation Date': 'string',
    'Operating Status': 'string',
    'Max Hourly HI Rate (mmBtu/hr)': 'float64',
    'Associated Generators & Nameplate Capacity (MWe)': 'string'
    })

# limit to columns we need
campd_facility = campd_facility[["Year", "State", 'Facility ID', 'Unit ID', "EPA Region", 'Source Category', 'Latitude',
       'Longitude', 'SO2 Phase', 'NOx Phase',
       'Commercial Operation Date', 'Operating Status',
       'Max Hourly HI Rate (mmBtu/hr)',
       'Associated Generators & Nameplate Capacity (MWe)']]

# filter out rows that are not in campd_ny, campd_wy, or campd_tx
campd_facility = campd_facility[
    campd_facility["Facility ID"].isin(campd_ny["Facility ID"]) |
    campd_facility["Facility ID"].isin(campd_wy["Facility ID"]) |
    campd_facility["Facility ID"].isin(campd_tx["Facility ID"])
]

In [ ]:
# function to extract and sum nameplate capacities in facility data (original data only has nameplate capacity in parentheses with associated generators)
def extract_and_sum_nameplate_capacity(data):
    data = str(data).strip()  # Convert data to string and remove leading/trailing whitespace
    matches = re.findall(r'\((\d+(?:\.\d+)?)\)', data) # find floats or integers in parentheses (can be float like 191.8 or integer like 170)

    if matches:
        total_sum = 0.0
        for match in matches:
            total_sum += float(match)
        #print("Total sum: ", total_sum)
        return total_sum
    else:
        #print("No matches found")
        return None

campd_facility['Nameplate Capacity (MWe)'] = campd_facility['Associated Generators & Nameplate Capacity (MWe)'].apply(extract_and_sum_nameplate_capacity)
campd_facility.to_csv('Facility/Concatted File/facility_2019_2024.csv', index=False)

In [62]:
campd_facility.isna().sum()

Year                                                   0
State                                                  0
Facility ID                                            0
Unit ID                                                0
EPA Region                                             0
Source Category                                        0
Latitude                                               0
Longitude                                              0
SO2 Phase                                           1530
NOx Phase                                           6366
Commercial Operation Date                              0
Operating Status                                      10
Max Hourly HI Rate (mmBtu/hr)                         19
Associated Generators & Nameplate Capacity (MWe)       0
Nameplate Capacity (MWe)                               0
dtype: int64

<h3>Merge Facility and Emission dfs</h3>

- merge facility and emission data based on their facility, unit id combination
- we look for any mismatches and conduct basic transformation to remove some of them (some units change their id seeminlgy routinely every year in either dataset by adding and removing leading zeroes. they are clearly the same since all other identifiers like facility id, associated generators etc are the same)

<h4>Wyoming</h4>

In [ ]:
# Look for mismatched Unit IDs between campd_facility and campd_wy for Wyoming facilities

# filter and get unique Facility-Unit pairs for both dfs
fac_units_1 = campd_facility[
    campd_facility["State"] == "WY"
][["Facility ID", "Unit ID"]].drop_duplicates()

fac_units_2 = campd_wy[["Facility ID", "Unit ID"]].drop_duplicates()

# outer merge with indicator (extra column that will show "both" if the pair exists in both dfs)
merged = fac_units_1.merge(fac_units_2, on=["Facility ID", "Unit ID"], how="outer", indicator=True)

# filter mismatches
mismatches = merged[merged["_merge"] != "both"]

# count mismatched units per facility
mismatch_counts = mismatches.groupby("Facility ID").size().reset_index(name="Mismatched Unit Count")

# include the mismatching unit IDs as list
mismatch_units = (
    mismatches.groupby("Facility ID")["Unit ID"]
    .apply(list)
    .reset_index(name="Mismatching Unit IDs")
)

# merge count and unit list
result = mismatch_counts.merge(mismatch_units, on="Facility ID")

display(result)


,Facility ID,Mismatched Unit Count,Mismatching Unit IDs
0,7504,1,[1]
1,55479,1,[1]
2,56319,2,"[001, 1]"
3,56596,1,[1]


In [13]:
# replace all unit IDs that are "001" or "01" with "1" to eliminate mismatches (see above)
campd_wy["Unit ID"] = campd_wy["Unit ID"].replace("001", "1")
campd_wy["Unit ID"] = campd_wy["Unit ID"].replace("01", "1")
campd_facility["Unit ID"] = campd_facility["Unit ID"].replace("001", "1")
campd_facility["Unit ID"] = campd_facility["Unit ID"].replace("01", "1")

In [14]:
# check after transformation

fac_units_1 = campd_facility[
    campd_facility["State"] == "WY"
][["Facility ID", "Unit ID"]].drop_duplicates()
fac_units_2 = campd_wy[["Facility ID", "Unit ID"]].drop_duplicates()

merged = fac_units_1.merge(fac_units_2, on=["Facility ID", "Unit ID"], how="outer", indicator=True)

mismatches = merged[merged["_merge"] != "both"]

mismatch_counts = mismatches.groupby("Facility ID").size().reset_index(name="Mismatched Unit Count")

mismatch_units = (
    mismatches.groupby("Facility ID")["Unit ID"]
    .apply(list)
    .reset_index(name="Mismatching Unit IDs")
)

result = mismatch_counts.merge(mismatch_units, on="Facility ID")

display(result)


,Facility ID,Mismatched Unit Count,Mismatching Unit IDs


In [15]:
campd_facility_wy = campd_facility[campd_facility["State"] == "WY"]
campd_facility_wy = campd_facility_wy[["Year", 'Facility ID', 'Unit ID', "EPA Region", 'Source Category', 'Latitude',
       'Longitude', 'SO2 Phase', 'NOx Phase',
       'Commercial Operation Date', 'Operating Status',
       'Max Hourly HI Rate (mmBtu/hr)',
       'Associated Generators & Nameplate Capacity (MWe)', 'Nameplate Capacity (MWe)']]
# add the year column to campd_wy for the join
campd_wy['Year'] = campd_wy['Date'].dt.year
campd_wy_facilities = pd.merge(campd_wy, campd_facility_wy, on=['Facility ID', 'Unit ID', 'Year'], how='left')

In [16]:
campd_wy_facilities.to_csv('WY/Hourly/Concatted File/wy_2019_2024_with_facility_Operating.csv', index=False)

<h4>New York</h4>

In [ ]:
# Look for mismatched Unit IDs between campd_facility and campd_ny

# filter and get unique Facility-Unit pairs for both dfs
fac_units_1 = campd_facility[
    campd_facility["State"] == "NY"
][["Facility ID", "Unit ID"]].drop_duplicates()

fac_units_2 = campd_ny[["Facility ID", "Unit ID"]].drop_duplicates()

# outer merge with indicator (extra column that will show "both" if the pair exists in both dfs)
merged = fac_units_1.merge(fac_units_2, on=["Facility ID", "Unit ID"], how="outer", indicator=True)

# filter mismatches
mismatches = merged[merged["_merge"] != "both"]

# count mismatched units per facility
mismatch_counts = mismatches.groupby("Facility ID").size().reset_index(name="Mismatched Unit Count")

# include the mismatching unit IDs as list
mismatch_units = (
    mismatches.groupby("Facility ID")["Unit ID"]
    .apply(list)
    .reset_index(name="Mismatching Unit IDs")
)

# merge count and unit list
result = mismatch_counts.merge(mismatch_units, on="Facility ID")

display(result)


,Facility ID,Mismatched Unit Count,Mismatching Unit IDs
0,2490,1,[CT0001]
1,2500,17,"[CT0001, CT0004, CT0005, CT0006, CT0007, CT000..."
2,2504,2,"[CT0001, CT0002]"
3,2511,8,"[U00004, U00005, U00006, U00007, U00008, U0000..."
4,2516,1,[UGT001]
5,2517,1,[UGT001]
6,2535,1,[2]
7,2679,1,[CT3]
8,7146,1,[UGT014]
9,7314,2,"[001, 1]"


In [18]:
# replace all unit IDs that are "001" or "01" with "1" to eliminate mismatches (see above)
campd_ny["Unit ID"] = campd_ny["Unit ID"].replace("001", "1")
campd_ny["Unit ID"] = campd_ny["Unit ID"].replace("01", "1")
campd_facility["Unit ID"] = campd_facility["Unit ID"].replace("001", "1")
campd_facility["Unit ID"] = campd_facility["Unit ID"].replace("01", "1")

In [ ]:
# here's an example of a facility that has a lot more entered units in facility data than in the emissions data

campd_facility_ny = campd_facility[campd_facility["State"] == "NY"]
display(campd_facility_ny[campd_facility_ny["Facility ID"] == 2500].drop_duplicates(subset=["Unit ID"])[["Facility ID","Unit ID"]])
campd_ny['Year'] = campd_ny['Date'].dt.year
display(campd_ny[campd_ny["Facility ID"] == 2500].drop_duplicates(subset=["Unit ID"])[["Facility ID","Unit ID"]])

,Facility ID,Unit ID
14,2500,10
15,2500,20
16,2500,30
17,2500,CT0001
18,2500,CT0004
19,2500,CT0005
20,2500,CT0006
21,2500,CT0007
22,2500,CT0008
23,2500,CT0009


,Facility ID,Unit ID
33050,2500,10
35240,2500,20
38637,2500,30
39291,2500,UCC001


In [ ]:
# check after transformation

# filter and get unique Facility-Unit pairs
fac_units_1 = campd_facility[
    campd_facility["State"] == "NY"
][["Facility ID", "Unit ID"]].drop_duplicates()

fac_units_2 = campd_ny[["Facility ID", "Unit ID"]].drop_duplicates()

# outer merge with indicator (extra column that will show "both" if the pair exists in both dfs)
merged = fac_units_1.merge(fac_units_2, on=["Facility ID", "Unit ID"], how="outer", indicator=True)

# filter mismatches
mismatches = merged[merged["_merge"] != "both"]

# count mismatched units per facility
mismatch_counts = mismatches.groupby("Facility ID").size().reset_index(name="Mismatched Unit Count")

# include the mismatching unit IDs as list
mismatch_units = (
    mismatches.groupby("Facility ID")["Unit ID"]
    .apply(list)
    .reset_index(name="Mismatching Unit IDs")
)

# merge count and unit list
result = mismatch_counts.merge(mismatch_units, on="Facility ID")

display(result)

# the 001 -> 1 transformation fixed only one unit mismatch


,Facility ID,Mismatched Unit Count,Mismatching Unit IDs
0,2490,1,[CT0001]
1,2500,17,"[CT0001, CT0004, CT0005, CT0006, CT0007, CT000..."
2,2504,2,"[CT0001, CT0002]"
3,2511,8,"[U00004, U00005, U00006, U00007, U00008, U0000..."
4,2516,1,[UGT001]
5,2517,1,[UGT001]
6,2535,1,[2]
7,2679,1,[CT3]
8,7146,1,[UGT014]
9,7869,1,[UGT011]


In [21]:
campd_facility_ny = campd_facility[campd_facility["State"] == "NY"]
campd_facility_ny = campd_facility_ny[["Year", 'Facility ID', 'Unit ID', "EPA Region", 'Source Category', 'Latitude',
       'Longitude', 'SO2 Phase', 'NOx Phase',
       'Commercial Operation Date', 'Operating Status',
       'Max Hourly HI Rate (mmBtu/hr)',
       'Associated Generators & Nameplate Capacity (MWe)', 'Nameplate Capacity (MWe)']]
# add the year column to campd_ny for the join
campd_ny['Year'] = campd_ny['Date'].dt.year
campd_ny_facilities = pd.merge(campd_ny, campd_facility_ny, on=['Facility ID', 'Unit ID', 'Year'], how='left')

In [ ]:
check = campd_ny_facilities.isna().sum()
check[check > 0]

# in the grand scheme of things, the number of facility-unit mismatches is not that high for total rows (33,324)

Associated Stacks                                   2038601
Gross Load (MW)                                      225011
Steam Load (1000 lb/hr)                             2176919
SO2 Rate (lbs/mmBtu)                                    673
SO2 Rate Measure Indicator                              673
CO2 Rate (short tons/mmBtu)                             673
CO2 Rate Measure Indicator                              673
NOx Rate (lbs/mmBtu)                                    577
NOx Rate Measure Indicator                              577
Secondary Fuel Type                                  610443
SO2 Controls                                        2354374
NOx Controls                                         440976
PM Controls                                         2133711
Hg Controls                                         2400701
EPA Region                                            33324
Source Category                                       33324
Latitude                                

In [23]:
campd_ny_facilities.to_csv('NY/Hourly/Concatted File/ny_2019_2024_with_facility_Operating.csv', index=False)

<h4>Texas</h4>

In [ ]:
# Look for mismatched Unit IDs between campd_facility and campd_tx

# filter and get unique Facility-Unit pairs
fac_units_1 = campd_facility[
    campd_facility["State"] == "TX"
][["Facility ID", "Unit ID"]].drop_duplicates()

fac_units_2 = campd_tx[["Facility ID", "Unit ID"]].drop_duplicates()

# outer merge with indicator (extra column that will show "both" if the pair exists in both dfs)
merged = fac_units_1.merge(fac_units_2, on=["Facility ID", "Unit ID"], how="outer", indicator=True)

# filter mismatches
mismatches = merged[merged["_merge"] != "both"]

# count mismatched units per facility
mismatch_counts = mismatches.groupby("Facility ID").size().reset_index(name="Mismatched Unit Count")

# include the mismatching unit IDs as list
mismatch_units = (
    mismatches.groupby("Facility ID")["Unit ID"]
    .apply(list)
    .reset_index(name="Mismatching Unit IDs")
)

# merge count and unit list
result = mismatch_counts.merge(mismatch_units, on="Facility ID")

display(result)


,Facility ID,Mismatched Unit Count,Mismatching Unit IDs
0,3459,1,[2]
1,3470,1,[CTSC]
2,3476,1,[4]
3,3548,8,"[GT-1A, GT-1B, GT-2A, GT-2B, GT-3A, GT-3B, GT-..."
4,6139,1,[2]
5,55064,2,"[002, 2]"


In [25]:
# replace all unit IDs that are "001" or "01" with "1" to eliminate mismatches (see above)
campd_tx["Unit ID"] = campd_tx["Unit ID"].replace("001", "1")
campd_tx["Unit ID"] = campd_tx["Unit ID"].replace("01", "1")
campd_facility["Unit ID"] = campd_facility["Unit ID"].replace("001", "1")
campd_facility["Unit ID"] = campd_facility["Unit ID"].replace("01", "1")

In [ ]:
# check after transformation

# filter and get unique Facility-Unit pairs
fac_units_1 = campd_facility[
    campd_facility["State"] == "TX"
][["Facility ID", "Unit ID"]].drop_duplicates()

fac_units_2 = campd_tx[["Facility ID", "Unit ID"]].drop_duplicates()

# outer merge with indicator (extra column that will show "both" if the pair exists in both dfs)
merged = fac_units_1.merge(fac_units_2, on=["Facility ID", "Unit ID"], how="outer", indicator=True)

# filter mismatches
mismatches = merged[merged["_merge"] != "both"]

# count mismatched units per facility
mismatch_counts = mismatches.groupby("Facility ID").size().reset_index(name="Mismatched Unit Count")

# include the mismatching unit IDs as list
mismatch_units = (
    mismatches.groupby("Facility ID")["Unit ID"]
    .apply(list)
    .reset_index(name="Mismatching Unit IDs")
)

# merge count and unit list
result = mismatch_counts.merge(mismatch_units, on="Facility ID")

display(result)

# the 001 -> 1 transformation fixed none here

,Facility ID,Mismatched Unit Count,Mismatching Unit IDs
0,3459,1,[2]
1,3470,1,[CTSC]
2,3476,1,[4]
3,3548,8,"[GT-1A, GT-1B, GT-2A, GT-2B, GT-3A, GT-3B, GT-..."
4,6139,1,[2]
5,55064,2,"[002, 2]"


In [27]:
campd_facility_tx = campd_facility[campd_facility["State"] == "TX"]
campd_facility_tx = campd_facility_tx[["Year", 'Facility ID', 'Unit ID', "EPA Region", 'Source Category', 'Latitude',
       'Longitude', 'SO2 Phase', 'NOx Phase',
       'Commercial Operation Date', 'Operating Status',
       'Max Hourly HI Rate (mmBtu/hr)',
       'Associated Generators & Nameplate Capacity (MWe)', 'Nameplate Capacity (MWe)']]
# add the year column to campd_ny for the join
campd_tx['Year'] = campd_tx['Date'].dt.year
campd_tx_facilities = pd.merge(campd_tx, campd_facility_tx, on=['Facility ID', 'Unit ID', 'Year'], how='left')

In [ ]:
check = campd_tx_facilities.isna().sum()
check[check > 0]

# again, the number of mismatches is not dramatic for overall rows (50188)

Associated Stacks                                   8845211
Gross Load (MW)                                       95371
Steam Load (1000 lb/hr)                             9166954
SO2 Rate (lbs/mmBtu)                                   1351
SO2 Rate Measure Indicator                             1351
CO2 Rate (short tons/mmBtu)                            1351
CO2 Rate Measure Indicator                             1351
NOx Rate (lbs/mmBtu)                                   1327
NOx Rate Measure Indicator                             1327
Secondary Fuel Type                                 6761944
SO2 Controls                                        8476610
NOx Controls                                         332251
PM Controls                                         8039292
Hg Controls                                         8509651
EPA Region                                            50188
Source Category                                       50188
Latitude                                

In [29]:
campd_tx_facilities.to_csv('TX/Hourly/Concatted File/tx_2019_2024_with_facility_Operating.csv', index=False)

<h2>Transform and Merge EIA-860 Data using crosswalk</h2>

(has additional data about e. g. seasonal generator nameplate capacities or age, but needs to be aggregated to a unit level)

- fill empty values with nan
- artificially extend to 2024 (data only goes up to 2023, but we can reasonably assume that this kind of data hasn't changed much over a single year)
- use the official crosswalk file that has predefined connections between CAMPD and EIA data (e. g. unit definitions differ)
- the original structure only has rows for when changes are documented, for a join with emissions data, we manually create yearly rows (copy earliest entries for each unit up to their latest change or the end date 2024)
- based on the common identifiers, we join the crosswalk data with the oiringal EIA-860 data
- aggregate, on a CAMPD unit basis, to calculate sum/min/max/mean for the relevant columns
- merge with the original emissions data 
- to evaluate consistency, measure average difference in recorded Nameplate Capacities for CAMPD and EIA. For all rows where the difference exceeds a threshold of 30 MW, we assume the EIA data to be inaccurate and set all of its values to nan.

In [ ]:
# read eia generator data
eia_generator = pd.read_csv('EIA/Concatted File/eia_generator_2019_2023.csv')

# these cols have blank strings (" ") that should be nan
columns_to_clean = [
    "Nameplate Power Factor", "Summer Capacity (MW)", "Winter Capacity (MW)",
    "Minimum Load (MW)", "Planned Retirement Year"
]

# replace blank strings with nan (any strings with only whitespace)
eia_generator[columns_to_clean] = eia_generator[columns_to_clean].replace(r'^\s*$', np.nan, regex=True)

# now define full dtype dictionary
eia_generator_type_dict = {
    "Year": "Int64", 
    "Utility ID": "string",
    "Utility Name": "string",
    "Plant Code": "Int64",  
    "Plant Name": "string",
    "State": "string",
    "Generator ID": "string",
    "Technology": "string",
    "Prime Mover": "string",
    "Nameplate Capacity (MW)": "float64",
    "Nameplate Power Factor": "float64", 
    "Summer Capacity (MW)": "float64",  
    "Winter Capacity (MW)": "float64",
    "Minimum Load (MW)": "float64",
    "Operating Year": "Int64",
    "Planned Retirement Year": "float64"
}

# filter needed cols
eia_generator = eia_generator[["Year","Utility ID","Utility Name", "Plant Code", "Plant Name", "State", "Generator ID", "Technology", "Prime Mover", "Nameplate Capacity (MW)", "Nameplate Power Factor", "Summer Capacity (MW)", "Winter Capacity (MW)", "Minimum Load (MW)", "Operating Year", "Planned Retirement Year"]]

# Artificially extend the unit_level dataframe to include 2024 data (just copy the 2023 data and set Year to 2024)
eia_generator_2023 = eia_generator[eia_generator["Year"] == 2023].copy()
eia_generator_2024 = eia_generator_2023.copy()
eia_generator_2024["Year"] = 2024
eia_generator = pd.concat([eia_generator, eia_generator_2024], ignore_index=True)

# cast all dtypes according to the dictionary
eia_generator = eia_generator.astype(eia_generator_type_dict)
eia_generator


,Year,Utility ID,Utility Name,Plant Code,Plant Name,State,Generator ID,Technology,Prime Mover,Nameplate Capacity (MW),Nameplate Power Factor,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Operating Year,Planned Retirement Year
0,2019,15474,Public Service Co of Oklahoma,127,Oklaunion,TX,1,Conventional Steam Coal,ST,720.0,0.90,650.0,650.0,215.0,1986,2020.0
1,2019,54888,NRG Texas Power LLC,298,Limestone,TX,1,Conventional Steam Coal,ST,893.0,0.94,831.0,831.0,300.0,1985,2030.0
2,2019,54888,NRG Texas Power LLC,298,Limestone,TX,2,Conventional Steam Coal,ST,956.8,0.92,858.0,858.0,300.0,1986,2030.0
3,2019,58971,Danskammer Energy,2480,Danskammer Generating Station,NY,1,Natural Gas Steam Turbine,ST,72.0,0.85,70.7,68.2,20.0,1951,NaN
4,2019,58971,Danskammer Energy,2480,Danskammer Generating Station,NY,2,Natural Gas Steam Turbine,ST,73.5,0.85,65.7,65.5,20.0,1954,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4181,2024,63082,ProEnergy Services,65373,Brotman Power Station,TX,CTG-2,Natural Gas Fired Combustion Turbine,GT,60.5,0.85,44.6,49.8,22.5,2023,NaN
4182,2024,63082,ProEnergy Services,65373,Brotman Power Station,TX,CTG-3,Natural Gas Fired Combustion Turbine,GT,60.5,0.85,44.6,49.8,22.5,2023,NaN
4183,2024,63082,ProEnergy Services,65373,Brotman Power Station,TX,CTG-4,Natural Gas Fired Combustion Turbine,GT,60.5,0.85,44.6,49.8,22.5,2023,NaN
4184,2024,63082,ProEnergy Services,65373,Brotman Power Station,TX,CTG-5,Natural Gas Fired Combustion Turbine,GT,60.5,0.85,44.6,49.8,22.5,2023,NaN


In [ ]:
crosswalk = pd.read_csv("epa_eia_crosswalk.csv")
crosswalk = crosswalk[(crosswalk["CAMD_STATE"] == "NY") | (crosswalk["CAMD_STATE"] == "TX") | (crosswalk["CAMD_STATE"] == "WY")] # filter for the states we need
crosswalk[crosswalk["CAMD_PLANT_ID"] == 2490]

# the col "CAMD_STATUS_DATE" gives the date where changes where last made for this plant id and unit id combo. we have to create new rows for every individual year up until the next change and so on with an additional col "Year"
# to match with the campd data
# join on CAMP_PLANT_ID, CAMD_UNIT_ID and add CAMD_NAMEPLATE_CAPACITY, CAMD_STATUS_DATE, EIA_NAMEPLATE_CAPACITY, EIA_BOILER_ID

,SEQUENCE_NUMBER,CAMD_STATE,CAMD_FACILITY_NAME,CAMD_PLANT_ID,CAMD_UNIT_ID,CAMD_GENERATOR_ID,CAMD_NAMEPLATE_CAPACITY,CAMD_FUEL_TYPE,CAMD_LATITUDE,CAMD_LONGITUDE,...,EIA_LATITUDE,EIA_LONGITUDE,EIA_RETIRE_YEAR,PLANT_ID_CHANGE_FLAG,MOD_EIA_PLANT_ID,MOD_EIA_BOILER_ID,MOD_EIA_GENERATOR_ID_BOILER,MOD_EIA_GENERATOR_ID_GEN,MATCH_TYPE_GEN,MATCH_TYPE_BOILER
1772,1773,NY,Arthur Kill,2490,20,2,376.2,Pipeline Natural Gas,40.5915,-74.2027,...,40.591564,-74.200035,0.0,0.0,2490.0,20,2,2,3_1_Generator (generators) match on plant and ...,6_1_EnviroAssoc (boilers and generators) match...
1773,1774,NY,Arthur Kill,2490,30,3,535.5,Pipeline Natural Gas,40.5915,-74.2027,...,40.591564,-74.200035,0.0,0.0,2490.0,30,3,3,3_1_Generator (generators) match on plant and ...,6_1_EnviroAssoc (boilers and generators) match...
1774,1775,NY,Arthur Kill,2490,CT0001,GT1,16.3,Pipeline Natural Gas,40.5915,-74.2027,...,40.591564,-74.200035,0.0,0.0,2490.0,NaN,NaN,GT1,3_1_Generator (generators) match on plant and ...,NaN


In [ ]:
# create yearly rows manually

crosswalk["CAMD_STATUS_DATE"] = pd.to_datetime(crosswalk["CAMD_STATUS_DATE"])

crosswalk_sorted = crosswalk.sort_values(by=["CAMD_PLANT_ID", "CAMD_UNIT_ID", "CAMD_STATUS_DATE"])

# Define end year up to which we generate rows
default_end_year = 2024

expanded_rows = [] # save rows for expansion

# Group by plant/unit and generate yearly rows
for (plant_id, unit_id), group in crosswalk_sorted.groupby(["CAMD_PLANT_ID", "CAMD_UNIT_ID"]):
    group = group.sort_values("CAMD_STATUS_DATE").reset_index(drop=True)

    for i in range(len(group)):
        start_year = group.loc[i, "CAMD_STATUS_DATE"].year
        end_year = (group.loc[i + 1, "CAMD_STATUS_DATE"].year 
                    if i + 1 < len(group) 
                    else default_end_year + 1)  # +1 so we include the final year

        for year in range(start_year, end_year):
            row = group.loc[i].copy()
            row["Year"] = year
            expanded_rows.append(row)

# Combine into new df
expanded_crosswalk = pd.DataFrame(expanded_rows)

expanded_crosswalk = expanded_crosswalk[expanded_crosswalk["Year"] >= 2019]
pd.set_option('display.max_columns', None)
expanded_crosswalk.sort_values(by=["CAMD_PLANT_ID", "CAMD_UNIT_ID", "Year"])

,SEQUENCE_NUMBER,CAMD_STATE,CAMD_FACILITY_NAME,CAMD_PLANT_ID,CAMD_UNIT_ID,CAMD_GENERATOR_ID,CAMD_NAMEPLATE_CAPACITY,CAMD_FUEL_TYPE,CAMD_LATITUDE,CAMD_LONGITUDE,CAMD_STATUS,CAMD_STATUS_DATE,CAMD_RETIRE_YEAR,MOD_CAMD_UNIT_ID,MOD_CAMD_GENERATOR_ID,EIA_STATE,EIA_PLANT_NAME,EIA_PLANT_ID,EIA_GENERATOR_ID,EIA_NAMEPLATE_CAPACITY,EIA_BOILER_ID,EIA_UNIT_TYPE,EIA_FUEL_TYPE,EIA_LATITUDE,EIA_LONGITUDE,EIA_RETIRE_YEAR,PLANT_ID_CHANGE_FLAG,MOD_EIA_PLANT_ID,MOD_EIA_BOILER_ID,MOD_EIA_GENERATOR_ID_BOILER,MOD_EIA_GENERATOR_ID_GEN,MATCH_TYPE_GEN,MATCH_TYPE_BOILER,Year
0,21,TX,Copper Station,9,CTG-1,1,70.0,Pipeline Natural Gas,31.7569,-106.375,OPR,1980-07-01,0,CTG-1,1,TX,Copper,9.0,1,86.9,NaN,GT,NG,31.7569,-106.375,0.0,0.0,9.0,NaN,NaN,1,3_1_Generator (generators) match on plant and ...,NaN,2019
0,21,TX,Copper Station,9,CTG-1,1,70.0,Pipeline Natural Gas,31.7569,-106.375,OPR,1980-07-01,0,CTG-1,1,TX,Copper,9.0,1,86.9,NaN,GT,NG,31.7569,-106.375,0.0,0.0,9.0,NaN,NaN,1,3_1_Generator (generators) match on plant and ...,NaN,2020
0,21,TX,Copper Station,9,CTG-1,1,70.0,Pipeline Natural Gas,31.7569,-106.375,OPR,1980-07-01,0,CTG-1,1,TX,Copper,9.0,1,86.9,NaN,GT,NG,31.7569,-106.375,0.0,0.0,9.0,NaN,NaN,1,3_1_Generator (generators) match on plant and ...,NaN,2021
0,21,TX,Copper Station,9,CTG-1,1,70.0,Pipeline Natural Gas,31.7569,-106.375,OPR,1980-07-01,0,CTG-1,1,TX,Copper,9.0,1,86.9,NaN,GT,NG,31.7569,-106.375,0.0,0.0,9.0,NaN,NaN,1,3_1_Generator (generators) match on plant and ...,NaN,2022
0,21,TX,Copper Station,9,CTG-1,1,70.0,Pipeline Natural Gas,31.7569,-106.375,OPR,1980-07-01,0,CTG-1,1,TX,Copper,9.0,1,86.9,NaN,GT,NG,31.7569,-106.375,0.0,0.0,9.0,NaN,NaN,1,3_1_Generator (generators) match on plant and ...,NaN,2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,6926,NY,Ravenswood Steam Plant,880100,BLR004,NaN,NaN,Residual Oil,40.7600,-73.750,OPR,1947-12-01,0,BLR004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Manual CAMD Excluded,Manual CAMD Excluded,2020
0,6926,NY,Ravenswood Steam Plant,880100,BLR004,NaN,NaN,Residual Oil,40.7600,-73.750,OPR,1947-12-01,0,BLR004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Manual CAMD Excluded,Manual CAMD Excluded,2021
0,6926,NY,Ravenswood Steam Plant,880100,BLR004,NaN,NaN,Residual Oil,40.7600,-73.750,OPR,1947-12-01,0,BLR004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Manual CAMD Excluded,Manual CAMD Excluded,2022
0,6926,NY,Ravenswood Steam Plant,880100,BLR004,NaN,NaN,Residual Oil,40.7600,-73.750,OPR,1947-12-01,0,BLR004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Manual CAMD Excluded,Manual CAMD Excluded,2023


In [ ]:
expanded_crosswalk["EIA_PLANT_ID"] = expanded_crosswalk["EIA_PLANT_ID"].astype("Int64")
expanded_crosswalk["EIA_GENERATOR_ID"] = expanded_crosswalk["EIA_GENERATOR_ID"].astype("string")
expanded_crosswalk["Year"] = expanded_crosswalk["Year"].astype("Int64")

In [ ]:
# join crosswalk with additional eia_generator data
combined_crosswalk = expanded_crosswalk.merge(eia_generator, how="left", left_on=["EIA_PLANT_ID", "EIA_GENERATOR_ID", "Year"], right_on=["Plant Code", "Generator ID", "Year"])

In [ ]:
combined_crosswalk = combined_crosswalk[['CAMD_PLANT_ID',
       'CAMD_UNIT_ID', 'CAMD_GENERATOR_ID', 'CAMD_NAMEPLATE_CAPACITY', 'CAMD_STATUS_DATE', 'CAMD_RETIRE_YEAR', 'EIA_PLANT_ID',
       'EIA_GENERATOR_ID', 'EIA_NAMEPLATE_CAPACITY', 'EIA_BOILER_ID', 'EIA_RETIRE_YEAR', 'Year', 'Utility ID', 'Utility Name', 'Plant Code', 'Plant Name',
       'State', 'Generator ID', 'Technology', 'Prime Mover',
       'Nameplate Capacity (MW)', 'Nameplate Power Factor',
       'Summer Capacity (MW)', 'Winter Capacity (MW)', 'Minimum Load (MW)',
       'Operating Year', 'Planned Retirement Year']]

In [ ]:
# aggregate relevant columns on a CAMD-unit level

agg_funcs = {
    'Nameplate Capacity (MW)': 'sum',
    'Summer Capacity (MW)': 'sum',
    'Winter Capacity (MW)': 'sum',
    'Minimum Load (MW)': 'sum',
    'Operating Year': ['min', 'max', 'mean'], # get operating year of the oldest (min) and newest (max) generator in the unit as well as the average across all gens
    'Planned Retirement Year': 'max', # planned retirement year is very rarely even given, we just take max
}

unit_level = combined_crosswalk.groupby(['Year', 'EIA_PLANT_ID', 'CAMD_UNIT_ID']).agg(agg_funcs).reset_index()

display(unit_level)

# flatten multi-index columns (only keep the aggregation operator (min, max, mean) for Operating Year)
unit_level.columns = [
    f"{col[0]}_{col[1]}" if col[0] == "Operating Year" else col[0]
    for col in unit_level.columns.to_flat_index()
]

# set data types again for the final unit_level dataframe
unit_level = unit_level.astype({
    'Year': 'Int64',
    'EIA_PLANT_ID': 'Int64',
    'CAMD_UNIT_ID': 'string',
    'Nameplate Capacity (MW)': 'float64',
    'Summer Capacity (MW)': 'float64',
    'Winter Capacity (MW)': 'float64',
    'Minimum Load (MW)': 'float64',
    'Operating Year_min': 'Int64',
    'Operating Year_max': 'Int64',
    'Operating Year_mean': 'Int64',
    'Planned Retirement Year': 'Int64'
})

unit_level

Year EIA_PLANT_ID CAMD_UNIT_ID Nameplate Capacity (MW)  \
                                                         sum   
0     2019          298         LIM1                   893.0   
1     2019          298         LIM2                   956.8   
2     2019         2480            1                    72.0   
3     2019         2480            2                    73.5   
4     2019         2480            3                   147.1   
...    ...          ...          ...                     ...   
3195  2024        60459          CT1                    60.5   
3196  2024        60459          CT2                    60.5   
3197  2024        60460          CT1                    60.5   
3198  2024        60460          CT2                    60.5   
3199  2024        60468         CT-1                   121.5   

     Summer Capacity (MW) Winter Capacity (MW) Minimum Load (MW)  \
                      sum                  sum               sum   
0                   831.0                831.0             300.0   
1                   858.0                858.0             300.0   
2                    70.7                 68.2              20.0   
3                    65.7                 65.5              20.0   
4                   137.7                137.1              40.0   
...                   ...                  ...               ...   
3195                 43.0                 46.0              20.0   
3196                 43.0                 46.0              20.0   
3197                 43.0                 46.0              20.0   
3198                 43.0                 46.0              20.0   
3199                117.0                119.0              25.0   

     Operating Year               Planned Retirement Year  
                min   max    mean                     max  
0              1985  1985  1985.0                  2030.0  
1              1986  1986  1986.0                  2030.0  
2              1951  1951  1951.0                     NaN  
3              1954  1954  1954.0                     NaN  
4              1959  1959  1959.0                     NaN  
...             ...   ...     ...                     ...  
3195           2017  2017  2017.0                     NaN  
3196           2017  2017  2017.0                     NaN  
3197           2017  2017  2017.0                     NaN  
3198           2017  2017  2017.0                     NaN  
3199           2018  2018  2018.0                     NaN  

[3200 rows x 11 columns]

,Year,EIA_PLANT_ID,CAMD_UNIT_ID,Nameplate Capacity (MW),Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Operating Year_min,Operating Year_max,Operating Year_mean,Planned Retirement Year
0,2019,298,LIM1,893.0,831.0,831.0,300.0,1985,1985,1985,2030
1,2019,298,LIM2,956.8,858.0,858.0,300.0,1986,1986,1986,2030
2,2019,2480,1,72.0,70.7,68.2,20.0,1951,1951,1951,<NA>
3,2019,2480,2,73.5,65.7,65.5,20.0,1954,1954,1954,<NA>
4,2019,2480,3,147.1,137.7,137.1,40.0,1959,1959,1959,<NA>
...,...,...,...,...,...,...,...,...,...,...,...
3195,2024,60459,CT1,60.5,43.0,46.0,20.0,2017,2017,2017,<NA>
3196,2024,60459,CT2,60.5,43.0,46.0,20.0,2017,2017,2017,<NA>
3197,2024,60460,CT1,60.5,43.0,46.0,20.0,2017,2017,2017,<NA>
3198,2024,60460,CT2,60.5,43.0,46.0,20.0,2017,2017,2017,<NA>


In [ ]:
# Calc min, max, and mean ages of the generators in the unit
unit_level['unit_age_min'] = unit_level["Year"] - unit_level['Operating Year_min']
unit_level['unit_age_max'] = unit_level["Year"] - unit_level['Operating Year_max']
unit_level['unit_age_mean'] = unit_level["Year"] - unit_level['Operating Year_mean']

In [ ]:
unit_level = unit_level.rename(columns={
    "EIA_PLANT_ID": "Facility ID",
    "CAMD_UNIT_ID": "Unit ID"
})

In [ ]:
# read in emission dfs
campd_ny = pd.read_csv("NY/Hourly/Concatted File/ny_2019_2024_with_facility_Operating.csv", dtype=campd_type_dict)
campd_wy = pd.read_csv("WY/Hourly/Concatted File/wy_2019_2024_with_facility_Operating.csv", type=campd_type_dict)
campd_tx = pd.read_csv("TX/Hourly/Concatted File/tx_2019_2024_with_facility_Operating.csv", type=campd_type_dict)

In [ ]:
# Merge each emission df with eia unit_level data
complete_campd_ny = pd.merge(campd_ny, unit_level, on=['Year', 'Facility ID', 'Unit ID'], how='left')
complete_campd_wy = pd.merge(campd_wy, unit_level, on=['Year', 'Facility ID', 'Unit ID'], how='left')
complete_campd_tx = pd.merge(campd_tx, unit_level, on=['Year', 'Facility ID', 'Unit ID'], how='left')

In [ ]:
# Calculate differences between aggregated EIA Nameplate Capacity and CAMPD Nameplate Capacity

# get abs difference
complete_campd_ny["Capacity Difference"] = (
    complete_campd_ny["Nameplate Capacity (MWe)"] - complete_campd_ny["Nameplate Capacity (MW)"]
).abs()

# drop rows with nans in either of the capacity columns
diff_df = complete_campd_ny.dropna(subset=["Nameplate Capacity (MWe)", "Nameplate Capacity (MW)"])

# calculate average difference
average_difference_ny = diff_df["Capacity Difference"].mean()
print(f"Average Capacity Difference NY: {average_difference_ny:.2f} MW")

# tx
complete_campd_tx["Capacity Difference"] = (
    complete_campd_tx["Nameplate Capacity (MWe)"] - complete_campd_tx["Nameplate Capacity (MW)"]
).abs()

diff_df = complete_campd_tx.dropna(subset=["Nameplate Capacity (MWe)", "Nameplate Capacity (MW)"])

average_difference_tx = diff_df["Capacity Difference"].mean()
print(f"Average Capacity Difference TX: {average_difference_tx:.2f} MW")

# wy
complete_campd_wy["Capacity Difference"] = (
    complete_campd_wy["Nameplate Capacity (MWe)"] - complete_campd_wy["Nameplate Capacity (MW)"]
).abs()

diff_df = complete_campd_wy.dropna(subset=["Nameplate Capacity (MWe)", "Nameplate Capacity (MW)"])

average_difference_wy = diff_df["Capacity Difference"].mean()
print(f"Average Capacity Difference WY: {average_difference_wy:.2f} MW")


print("Average Difference across all: ", np.mean([average_difference_ny, average_difference_tx, average_difference_wy]))

Average Capacity Difference NY: 80.30 MW
Average Capacity Difference TX: 120.39 MW
Average Capacity Difference WY: 42.51 MW
Average Difference across all:  81.06707020834027


In [ ]:
# fill EIA columns with na if it capacity difference exceeds a certain threshold

# rename the EIA column to make it clearer
for df in [complete_campd_ny, complete_campd_wy, complete_campd_tx]:
    df.rename(columns={
        "Nameplate Capacity (MW)": "Nameplate Capacity (MW) - EIA"
    }, inplace=True)

threshold = 30 # set threshold at 30 MW difference
columns_to_null = [
    "Nameplate Capacity (MW) - EIA", 
    "Summer Capacity (MW)",
    "Winter Capacity (MW)",
    "Minimum Load (MW)"
]

# Recalculate capacity difference
complete_campd_wy["Capacity Difference"] = (
    complete_campd_wy["Nameplate Capacity (MWe)"] - complete_campd_wy["Nameplate Capacity (MW) - EIA"]
).abs()

# Identify rows with unreasonable differences
condition = complete_campd_wy["Capacity Difference"] > threshold

# Set selected columns to NaN where condition is met
complete_campd_wy.loc[condition, columns_to_null] = np.nan


# Recalculate capacity difference
complete_campd_tx["Capacity Difference"] = (
    complete_campd_tx["Nameplate Capacity (MWe)"] - complete_campd_tx["Nameplate Capacity (MW) - EIA"]
).abs()

# Identify rows with unreasonable differences
condition = complete_campd_tx["Capacity Difference"] > threshold

# Set selected columns to NaN where condition is met
complete_campd_tx.loc[condition, columns_to_null] = np.nan

# Recalculate capacity difference
complete_campd_ny["Capacity Difference"] = (
    complete_campd_ny["Nameplate Capacity (MWe)"] - complete_campd_ny["Nameplate Capacity (MW) - EIA"]
).abs()

# Identify rows with unreasonable differences
condition = complete_campd_ny["Capacity Difference"] > threshold

# Set selected columns to NaN where condition is met
complete_campd_ny.loc[condition, columns_to_null] = np.nan

In [ ]:
# Calculate capacity difference again for comparison

# Create a column for absolute difference
complete_campd_ny["Capacity Difference"] = (
    complete_campd_ny["Nameplate Capacity (MWe)"] - complete_campd_ny["Nameplate Capacity (MW) - EIA"]
).abs()

# Drop rows with NaNs in either of the capacity columns
diff_df = complete_campd_ny.dropna(subset=["Nameplate Capacity (MWe)", "Nameplate Capacity (MW) - EIA"])

# Calculate average difference
average_difference_ny = diff_df["Capacity Difference"].mean()
print(f"Average Capacity Difference NY: {average_difference_ny:.2f} MW")

# Create a column for absolute difference
complete_campd_tx["Capacity Difference"] = (
    complete_campd_tx["Nameplate Capacity (MWe)"] - complete_campd_tx["Nameplate Capacity (MW) - EIA"]
).abs()

# Drop rows with NaNs in either of the capacity columns
diff_df = complete_campd_tx.dropna(subset=["Nameplate Capacity (MWe)", "Nameplate Capacity (MW) - EIA"])

# Calculate average difference
average_difference_tx = diff_df["Capacity Difference"].mean()
print(f"Average Capacity Difference TX: {average_difference_tx:.2f} MW")

# Create a column for absolute difference
complete_campd_wy["Capacity Difference"] = (
    complete_campd_wy["Nameplate Capacity (MWe)"] - complete_campd_wy["Nameplate Capacity (MW) - EIA"]
).abs()

# Drop rows with NaNs in either of the capacity columns
diff_df = complete_campd_wy.dropna(subset=["Nameplate Capacity (MWe)", "Nameplate Capacity (MW) - EIA"])

# Calculate average difference
average_difference_wy = diff_df["Capacity Difference"].mean()
print(f"Average Capacity Difference WY: {average_difference_wy:.2f} MW")


print("Average Difference across all: ", np.mean([average_difference_ny, average_difference_tx, average_difference_wy]))



Average Capacity Difference NY: 2.24 MW
Average Capacity Difference TX: 3.57 MW
Average Capacity Difference WY: 13.11 MW
Average Difference across all:  6.306405955194138


In [ ]:
complete_campd_ny.to_csv('NY/Hourly/Concatted File/ny_2019_2024_complete_Operating.csv', index=False)

In [ ]:
complete_campd_wy.to_csv('WY/Hourly/Concatted File/wy_2019_2024_complete_Operating.csv', index=False)

In [ ]:
complete_campd_tx.to_csv('TX/Hourly/Concatted File/tx_2019_2024_complete_Operating.csv', index=False)

<h2>Data Cleaning of Final Datasets</h2>

- define all relevant columns and broad data types
- extract month and day columns from date
- handle null values (e. g. remove steam load where always null, fill categorical nulls logically -> if it's likely that lack of data was purposefully reported then fill with "None" e. g. no controls installed, if data is missing due to a lack of reports then fill with "Missing" e. g. Associated Generators)
- remove the EIA nameplate capacity (we assume CAMPD capacity to be more accurate)
- add calculated columns for hourly_capacity_factor and hourly_ramping_factor

In [4]:
# relevant columns
complete_campd_columns = [
    "State",
    "Facility Name",
    "Facility ID",
    "Unit ID",
    "Gross Load (MW)",
    "Steam Load (1000 lb/hr)",
    "SO2 Mass (lbs)",
    "SO2 Mass Measure Indicator",
    "SO2 Rate (lbs/mmBtu)",
    "SO2 Rate Measure Indicator",
    "CO2 Mass (short tons)",
    "CO2 Mass Measure Indicator",
    "CO2 Rate (short tons/mmBtu)",
    "CO2 Rate Measure Indicator",
    "NOx Mass (lbs)",
    "NOx Mass Measure Indicator",
    "NOx Rate (lbs/mmBtu)",
    "NOx Rate Measure Indicator",
    "Heat Input (mmBtu)",
    "Heat Input Measure Indicator",
    "Primary Fuel Type",
    "Secondary Fuel Type",
    "Unit Type",
    "SO2 Controls",
    "NOx Controls",
    "Program Code",
    "Hour",
    "Year",
    "Date",
    "Operating Time",
    "Source Category",
    "Latitude",
    "Longitude",
    "SO2 Phase",
    "NOx Phase",
    "Operating Status",
    "Max Hourly HI Rate (mmBtu/hr)",
    "Associated Generators & Nameplate Capacity (MWe)",
    "Nameplate Capacity (MWe)",
    "Nameplate Capacity (MW) - EIA",
    "Summer Capacity (MW)",
    "Winter Capacity (MW)",
    "Minimum Load (MW)",
    "Planned Retirement Year",
    "unit_age_min",
    "unit_age_max",
    "unit_age_mean"
]

# Filtered dtype dictionary
complete_campd_type_dict = {
    "State": "string",
    "Facility Name": "string",
    "Facility ID": "Int64",
    "Unit ID": "string",
    "Gross Load (MW)": "float64",
    "Steam Load (1000 lb/hr)": "float64",
    "SO2 Mass (lbs)": "float64",
    "SO2 Mass Measure Indicator": "string",
    "SO2 Rate (lbs/mmBtu)": "float64",
    "SO2 Rate Measure Indicator": "string",
    "CO2 Mass (short tons)": "float64",
    "CO2 Mass Measure Indicator": "string",
    "CO2 Rate (short tons/mmBtu)": "float64",
    "CO2 Rate Measure Indicator": "string",
    "NOx Mass (lbs)": "float64",
    "NOx Mass Measure Indicator": "string",
    "NOx Rate (lbs/mmBtu)": "float64",
    "NOx Rate Measure Indicator": "string",
    "Heat Input (mmBtu)": "float64",
    "Heat Input Measure Indicator": "string",
    "Primary Fuel Type": "string",
    "Secondary Fuel Type": "string",
    "Unit Type": "string",
    "SO2 Controls": "string",
    "NOx Controls": "string",
    "Program Code": "string",
    "Hour": "Int64",
    "Year": "Int64",
    "Operating Time": "float64",
    "Source Category": "string",
    "Latitude": "float64",
    "Longitude": "float64",
    "SO2 Phase": "string",
    "NOx Phase": "string",
    "Operating Status": "string",
    "Max Hourly HI Rate (mmBtu/hr)": "float64",
    "Associated Generators & Nameplate Capacity (MWe)": "string",
    "Nameplate Capacity (MW)": "float64",
    "Nameplate Capacity (MW) - EIA": "float64",
    "Summer Capacity (MW)": "float64",
    "Winter Capacity (MW)": "float64",
    "Minimum Load (MW)": "float64",
    "Planned Retirement Year": "Int64",
    "unit_age_min": "Int64",
    "unit_age_max": "Int64",
    "unit_age_mean": "float64"
}


<h3>Wyoming</h3>

In [ ]:
complete_campd_wy = pd.read_csv(
    "WY/Hourly/Concatted File/wy_2019_2024_complete_Operating.csv",  
    usecols=complete_campd_columns,
    dtype=complete_campd_type_dict,
    parse_dates=["Date"]
)

# extract month and day as new col from date
complete_campd_wy["Month"] = complete_campd_wy["Date"].dt.month
complete_campd_wy["Day"] = complete_campd_wy["Date"].dt.month

# create full datetime column (date + hour)
campd_wy["datetime"] = pd.to_datetime({
    'year': campd_wy["Date"].dt.year,
    'month': campd_wy["Date"].dt.month,
    'day': campd_wy["Date"].dt.day,
    'hour': campd_wy["Hour"]
})

In [50]:
print(complete_campd_wy.shape)
null_wy = complete_campd_wy.isnull().sum()
null_wy = null_wy[null_wy > 0].sort_values(ascending=False)
display(null_wy)

# Give null vals as percentage of total rows
total_rows = complete_campd_wy.shape[0]
null_percentages = (null_wy / total_rows) * 100
null_percentages.sort_values(ascending=False)

(993428, 49)


Steam Load (1000 lb/hr)          993428
Secondary Fuel Type              900646
Planned Retirement Year          616535
Nameplate Capacity (MW) - EIA    486966
Summer Capacity (MW)             486966
Winter Capacity (MW)             486966
Minimum Load (MW)                486966
NOx Phase                        325236
SO2 Controls                     216641
NOx Controls                      95841
dtype: int64

Steam Load (1000 lb/hr)          100.000000
Secondary Fuel Type               90.660420
Planned Retirement Year           62.061367
Nameplate Capacity (MW) - EIA     49.018751
Summer Capacity (MW)              49.018751
Winter Capacity (MW)              49.018751
Minimum Load (MW)                 49.018751
NOx Phase                         32.738759
SO2 Controls                      21.807418
NOx Controls                       9.647503
dtype: float64

In [ ]:
# handle null values appropriately

# remove Steam Load columns (always null)
complete_campd_wy = complete_campd_wy.drop(columns=["Steam Load (1000 lb/hr)"])

# we drop nameplate capacity (MW) - EIA (redundant with Nameplate Capacity (MWe) and has many nulls)
complete_campd_wy = complete_campd_wy.drop(columns=["Nameplate Capacity (MW) - EIA"])

# replace nan values in categorical columns with "Nothing"
cat_cols_with_na = ["Secondary Fuel Type", "NOx Phase", 
                    "SO2 Controls", "NOx Controls",
]
for col in cat_cols_with_na:
    complete_campd_wy[col] = complete_campd_wy[col].fillna("Nothing").astype("string")

print("Null values in complete_campd_wy after adjustments:")
null_wy = complete_campd_wy.isnull().sum()
print(null_wy[null_wy > 0].sort_values(ascending=False))

Null values in complete_campd_wy after adjustments:
Planned Retirement Year    616535
Summer Capacity (MW)       486966
Winter Capacity (MW)       486966
Minimum Load (MW)          486966
dtype: int64


In [52]:
# measurement methods
print("Measurement methods in complete_campd_wy:")
display(complete_campd_wy.groupby("CO2 Mass Measure Indicator")["CO2 Mass (short tons)"].count())
display(complete_campd_wy.groupby("NOx Mass Measure Indicator")["NOx Mass Measure Indicator"].count())
display(complete_campd_wy.groupby("SO2 Mass Measure Indicator")["SO2 Mass Measure Indicator"].count())


Measurement methods in complete_campd_wy:


CO2 Mass Measure Indicator
Measured                   983249
Measured and Substitute      9733
Substitute                    446
Name: CO2 Mass (short tons), dtype: int64

NOx Mass Measure Indicator
Calculated    993428
Name: NOx Mass Measure Indicator, dtype: int64

SO2 Mass Measure Indicator
Measured                   983624
Measured and Substitute      9210
Substitute                    594
Name: SO2 Mass Measure Indicator, dtype: int64

In [53]:
# calculate hourly capacity factor
complete_campd_wy["hourly_capacity_factor"] = (
    complete_campd_wy["Gross Load (MW)"] * complete_campd_wy["Operating Time"]
) / complete_campd_wy["Nameplate Capacity (MWe)"]

complete_campd_wy.head(5)

,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),SO2 Mass (lbs),SO2 Mass Measure Indicator,...,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean,Month,Day,hourly_capacity_factor
0,WY,Dave Johnston,4158,BW41,2019-01-01,0,1.0,103.0,728.4,Measured,...,93.0,99.0,45.0,<NA>,60,60,60.0,1,1,0.971698
1,WY,Dave Johnston,4158,BW41,2019-01-01,1,1.0,103.0,705.2,Measured,...,93.0,99.0,45.0,<NA>,60,60,60.0,1,1,0.971698
2,WY,Dave Johnston,4158,BW41,2019-01-01,2,1.0,103.0,699.8,Measured,...,93.0,99.0,45.0,<NA>,60,60,60.0,1,1,0.971698
3,WY,Dave Johnston,4158,BW41,2019-01-01,3,1.0,104.0,702.5,Measured,...,93.0,99.0,45.0,<NA>,60,60,60.0,1,1,0.981132
4,WY,Dave Johnston,4158,BW41,2019-01-01,4,1.0,103.0,719.9,Measured,...,93.0,99.0,45.0,<NA>,60,60,60.0,1,1,0.971698


In [ ]:
# calculate hourly ramping factor

# first creae the full hourly dataset by adding the nonoperating dataset back
filtered_out_rows_campd_wy = pd.read_csv('WY/Hourly/Concatted File/wy_2019_2024_NonOperating.csv', dtype=campd_type_dict, parse_dates=["Date"])
filtered_out_rows_campd_wy = filtered_out_rows_campd_wy[["Facility ID", "Unit ID", "Date", "Hour", "Gross Load (MW)"]]
filtered_out_rows_campd_wy["Year"] = filtered_out_rows_campd_wy["Date"].dt.year
facility_small = pd.read_csv('Facility/Concatted File/facility_2014_2024.csv', dtype={"Facility ID": "Int64", "Unit ID": "string", "Year": "Int64"})
facility_small = facility_small[["Year", 'Facility ID', 'Unit ID', "Nameplate Capacity (MWe)"]]
all_hours_wy = filtered_out_rows_campd_wy.merge(facility_small, on=['Year', 'Facility ID', 'Unit ID'], how='left')
all_hours_wy["Gross Load (MW)"] = all_hours_wy["Gross Load (MW)"].fillna(0)

all_hours_wy = pd.concat([all_hours_wy, complete_campd_wy[["Facility ID", "Unit ID", "Date", "Hour", "Gross Load (MW)", "Year", "Nameplate Capacity (MWe)"]]], ignore_index=True)

# Sort data by unit and time to calculate t-1 values
all_hours_wy = all_hours_wy.sort_values(["Facility ID", "Unit ID", "Date", "Hour"])

# Calculate lagged gross load per unit (get previous row's value for gross load)
all_hours_wy["gross_load_lag1"] = all_hours_wy.groupby(["Facility ID", "Unit ID"])["Gross Load (MW)"].shift(1)

# calc ramping factor
all_hours_wy["hourly_ramping_factor"] = (
    all_hours_wy["Gross Load (MW)"] - all_hours_wy["gross_load_lag1"]
) / all_hours_wy["Nameplate Capacity (MWe)"]

display(all_hours_wy.head(10))

complete_campd_wy = complete_campd_wy.merge(all_hours_wy[["Facility ID", "Unit ID", "Date", "Hour", "hourly_ramping_factor"]],
                       on=["Facility ID", "Unit ID", "Date", "Hour"],
                       how="left")
complete_campd_wy.head(5)


,Facility ID,Unit ID,Date,Hour,Gross Load (MW),Year,Nameplate Capacity (MWe),gross_load_lag1,hourly_ramping_factor
321772,4158,BW41,2019-01-01,0,103.0,2019,106.0,NaN,NaN
321773,4158,BW41,2019-01-01,1,103.0,2019,106.0,103.0,0.000000
321774,4158,BW41,2019-01-01,2,103.0,2019,106.0,103.0,0.000000
321775,4158,BW41,2019-01-01,3,104.0,2019,106.0,103.0,0.009434
321776,4158,BW41,2019-01-01,4,103.0,2019,106.0,104.0,-0.009434
321777,4158,BW41,2019-01-01,5,103.0,2019,106.0,103.0,0.000000
321778,4158,BW41,2019-01-01,6,103.0,2019,106.0,103.0,0.000000
321779,4158,BW41,2019-01-01,7,104.0,2019,106.0,103.0,0.009434
321780,4158,BW41,2019-01-01,8,104.0,2019,106.0,104.0,0.000000
321781,4158,BW41,2019-01-01,9,103.0,2019,106.0,104.0,-0.009434


,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),SO2 Mass (lbs),SO2 Mass Measure Indicator,...,Winter Capacity (MW),Minimum Load (MW),Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean,Month,Day,hourly_capacity_factor,hourly_ramping_factor
0,WY,Dave Johnston,4158,BW41,2019-01-01,0,1.0,103.0,728.4,Measured,...,99.0,45.0,<NA>,60,60,60.0,1,1,0.971698,NaN
1,WY,Dave Johnston,4158,BW41,2019-01-01,1,1.0,103.0,705.2,Measured,...,99.0,45.0,<NA>,60,60,60.0,1,1,0.971698,0.000000
2,WY,Dave Johnston,4158,BW41,2019-01-01,2,1.0,103.0,699.8,Measured,...,99.0,45.0,<NA>,60,60,60.0,1,1,0.971698,0.000000
3,WY,Dave Johnston,4158,BW41,2019-01-01,3,1.0,104.0,702.5,Measured,...,99.0,45.0,<NA>,60,60,60.0,1,1,0.981132,0.009434
4,WY,Dave Johnston,4158,BW41,2019-01-01,4,1.0,103.0,719.9,Measured,...,99.0,45.0,<NA>,60,60,60.0,1,1,0.971698,-0.009434


In [ ]:
complete_campd_wy.to_csv('WY/Hourly/Concatted File/wy_2019_2024_complete_cleaned_Operating.csv', index=False)

<h3>New York</h3>

In [ ]:
complete_campd_ny = pd.read_csv(
    'NY/Hourly/Concatted File/ny_2019_2024_complete_Operating.csv',  
    usecols=complete_campd_columns,
    dtype=complete_campd_type_dict,
    parse_dates=["Date"]
)

# extract day and month as new col from date
complete_campd_ny["Month"] = complete_campd_ny["Date"].dt.month
complete_campd_ny["Day"] = complete_campd_ny["Date"].dt.month

# create full datetime column (date + hour)
campd_ny["datetime"] = pd.to_datetime({
    'year': campd_ny["Date"].dt.year,
    'month': campd_ny["Date"].dt.month,
    'day': campd_ny["Date"].dt.day,
    'hour': campd_ny["Hour"]
})

In [57]:
print(complete_campd_ny.shape)
null_ny = complete_campd_ny.isnull().sum()
null_ny = null_ny[null_ny > 0].sort_values(ascending=False)
display(null_ny)

# Give null vals as percentage of total rows
total_rows = complete_campd_ny.shape[0]
null_percentages = (null_ny / total_rows) * 100
null_percentages.sort_values(ascending=False)

(2400701, 49)


Planned Retirement Year                             2392161
SO2 Controls                                        2354374
NOx Phase                                           2351870
Steam Load (1000 lb/hr)                             2176919
Secondary Fuel Type                                  610443
Minimum Load (MW)                                    489352
Winter Capacity (MW)                                 489352
Summer Capacity (MW)                                 489352
Nameplate Capacity (MW) - EIA                        489352
NOx Controls                                         440976
SO2 Phase                                            353102
unit_age_max                                         310577
unit_age_min                                         310577
unit_age_mean                                        310577
Gross Load (MW)                                      225011
Longitude                                             33324
Latitude                                

Planned Retirement Year                             99.644271
SO2 Controls                                        98.070272
NOx Phase                                           97.965969
Steam Load (1000 lb/hr)                             90.678473
Secondary Fuel Type                                 25.427698
Minimum Load (MW)                                   20.383713
Winter Capacity (MW)                                20.383713
Summer Capacity (MW)                                20.383713
Nameplate Capacity (MW) - EIA                       20.383713
NOx Controls                                        18.368635
SO2 Phase                                           14.708287
unit_age_max                                        12.936930
unit_age_min                                        12.936930
unit_age_mean                                       12.936930
Gross Load (MW)                                      9.372721
Longitude                                            1.388095
Latitude

In [ ]:
# handle categorical null values appropriately
cat_cols_with_na_none = [
    "SO2 Controls",
    "NOx Phase",
    "Secondary Fuel Type",
    "NOx Controls",
    "SO2 Phase"
]

cat_cols_with_na_unknown = [
    "Operating Status",
    "Associated Generators & Nameplate Capacity (MWe)",
    "Source Category", "Program Code", "CO2 Rate Measure Indicator",
    "SO2 Rate Measure Indicator",
    "NOx Rate Measure Indicator"
]

# here nan means that no data was likely purposely reported -> fill with "None"
for col in cat_cols_with_na_none:
    complete_campd_ny[col] = complete_campd_ny[col].fillna("Nothing").astype("string")

# here nan means that no data was likely reported -> fill with "Missing"
for col in cat_cols_with_na_unknown:
    complete_campd_ny[col] = complete_campd_ny[col].fillna("Missing").astype("string")

num_cols_with_na = [
    "Planned Retirement Year",
    "Steam Load (1000 lb/hr)",
    "unit_age_max",
    "unit_age_min",
    "unit_age_mean",
    "Minimum Load (MW)",
    "Winter Capacity (MW)",
    "Summer Capacity (MW)",
    #"Nameplate Capacity (MW) - EIA", # dropped later
    "Gross Load (MW)",
    "Longitude",
    "Latitude",
    "Max Hourly HI Rate (mmBtu/hr)",
    "Nameplate Capacity (MWe)",
    "CO2 Rate (short tons/mmBtu)",
    "SO2 Rate (lbs/mmBtu)",
    "NOx Rate (lbs/mmBtu)"
]

# we drop nameplate capacity (MW) - EIA (redundant with Nameplate Capacity (MWe) and has many nulls)
complete_campd_ny = complete_campd_ny.drop(columns=["Nameplate Capacity (MW) - EIA"])

# for now we keep it, let the models handle it
print("Null value percentages in complete_campd_ny after adjustments:")
total_rows = complete_campd_ny.shape[0]
null_ny = complete_campd_ny.isnull().sum()
null_ny = null_ny[null_ny > 0].sort_values(ascending=False)
null_percentages = (null_ny / total_rows) * 100
null_percentages.sort_values(ascending=False)



Null value percentages in complete_campd_ny after adjustments:


Planned Retirement Year          99.644271
Steam Load (1000 lb/hr)          90.678473
Summer Capacity (MW)             20.383713
Minimum Load (MW)                20.383713
Winter Capacity (MW)             20.383713
unit_age_mean                    12.936930
unit_age_max                     12.936930
unit_age_min                     12.936930
hourly_ramping_factor            10.761773
hourly_capacity_factor           10.760815
Gross Load (MW)                   9.372721
Nameplate Capacity (MWe)          1.388095
Max Hourly HI Rate (mmBtu/hr)     1.388095
Longitude                         1.388095
Latitude                          1.388095
CO2 Rate (short tons/mmBtu)       0.028033
SO2 Rate (lbs/mmBtu)              0.028033
NOx Rate (lbs/mmBtu)              0.024035
dtype: float64

In [65]:
# measurement methods
print("Measurement methods in complete_campd_ny:")
display(complete_campd_ny.groupby("CO2 Mass Measure Indicator")["CO2 Mass (short tons)"].count())
display(complete_campd_ny.groupby("NOx Mass Measure Indicator")["NOx Mass Measure Indicator"].count())
display(complete_campd_ny.groupby("SO2 Mass Measure Indicator")["SO2 Mass Measure Indicator"].count())


Measurement methods in complete_campd_ny:


CO2 Mass Measure Indicator
LME                          72482
Measured                   2251406
Measured and Substitute      76386
Substitute                     427
Name: CO2 Mass (short tons), dtype: int64

NOx Mass Measure Indicator
Calculated                  186287
LME                          72482
Measured                   2035031
Measured and Substitute     106474
Substitute                     427
Name: NOx Mass Measure Indicator, dtype: int64

SO2 Mass Measure Indicator
LME                          72482
Measured                   2280325
Measured and Substitute      16164
Substitute                   31730
Name: SO2 Mass Measure Indicator, dtype: int64

In [ ]:
# calculate hourly capacity factor
complete_campd_ny["hourly_capacity_factor"] = (
    complete_campd_ny["Gross Load (MW)"] * complete_campd_ny["Operating Time"]
) / complete_campd_ny["Nameplate Capacity (MWe)"]

complete_campd_ny.head(5)

,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),Steam Load (1000 lb/hr),SO2 Mass (lbs),...,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean,Month,Day,hourly_capacity_factor
0,NY,Danskammer Generating Station,2480,1,2019-06-04,11,0.08,0.0,NaN,0.000,...,70.7,68.2,20.0,<NA>,68,68,68.0,6,6,0.0
1,NY,Danskammer Generating Station,2480,1,2019-06-10,20,0.72,0.0,NaN,0.072,...,70.7,68.2,20.0,<NA>,68,68,68.0,6,6,0.0
2,NY,Danskammer Generating Station,2480,1,2019-06-10,21,1.00,0.0,NaN,0.100,...,70.7,68.2,20.0,<NA>,68,68,68.0,6,6,0.0
3,NY,Danskammer Generating Station,2480,1,2019-06-10,22,1.00,0.0,NaN,0.100,...,70.7,68.2,20.0,<NA>,68,68,68.0,6,6,0.0
4,NY,Danskammer Generating Station,2480,1,2019-06-10,23,1.00,0.0,NaN,0.100,...,70.7,68.2,20.0,<NA>,68,68,68.0,6,6,0.0


In [ ]:
# calculate hourly ramping factor
filtered_out_rows_campd_ny = pd.read_csv('NY/Hourly/Concatted File/ny_2019_2024_NonOperating.csv', dtype=campd_type_dict, parse_dates=["Date"])
filtered_out_rows_campd_ny["Year"] = filtered_out_rows_campd_ny["Date"].dt.year
facility_small = pd.read_csv('Facility/Concatted File/facility_2014_2024.csv', dtype={"Facility ID": "Int64", "Unit ID": "string", "Year": "Int64"})
facility_small = facility_small[["Year", 'Facility ID', 'Unit ID', "Nameplate Capacity (MWe)"]]
all_hours_ny = filtered_out_rows_campd_ny.merge(facility_small, on=['Year', 'Facility ID', 'Unit ID'], how='left')
all_hours_ny["Gross Load (MW)"] = all_hours_ny["Gross Load (MW)"].fillna(0)

all_hours_ny = pd.concat([all_hours_ny, complete_campd_ny[["Facility ID", "Unit ID", "Date", "Hour", "Gross Load (MW)", "Year", "Nameplate Capacity (MWe)"]]], ignore_index=True)

# Sort data by unit and time to calculate t-1 values
all_hours_ny = all_hours_ny.sort_values(["Facility ID", "Unit ID", "Date", "Hour"])

# Calculate lagged gross load per unit (get previous row's value for gross load)
all_hours_ny["gross_load_lag1"] = all_hours_ny.groupby(["Facility ID", "Unit ID"])["Gross Load (MW)"].shift(1)

# calc ramping factor
all_hours_ny["hourly_ramping_factor"] = (
    all_hours_ny["Gross Load (MW)"] - all_hours_ny["gross_load_lag1"]
) / all_hours_ny["Nameplate Capacity (MWe)"]

display(all_hours_ny.head(10))

complete_campd_ny = complete_campd_ny.merge(all_hours_ny[["Facility ID", "Unit ID", "Date", "Hour", "hourly_ramping_factor"]],
                        on=["Facility ID", "Unit ID", "Date", "Hour"],
                        how="left")
complete_campd_ny.head(5)


,State,Facility Name,Facility ID,Unit ID,Associated Stacks,Date,Hour,Operating Time,Gross Load (MW),Steam Load (1000 lb/hr),...,Unit Type,SO2 Controls,NOx Controls,PM Controls,Hg Controls,Program Code,Year,Nameplate Capacity (MWe),gross_load_lag1,hourly_ramping_factor
0,NY,Danskammer Generating Station,2480,1,<NA>,2019-01-01,0,0.0,0.0,NaN,...,Tangentially-fired,<NA>,<NA>,<NA>,<NA>,"ARP, CSNOX, CSOSG2, CSSO2G1, RGGI",2019,72.0,NaN,NaN
1,NY,Danskammer Generating Station,2480,1,<NA>,2019-01-01,1,0.0,0.0,NaN,...,Tangentially-fired,<NA>,<NA>,<NA>,<NA>,"ARP, CSNOX, CSOSG2, CSSO2G1, RGGI",2019,72.0,0.0,0.0
2,NY,Danskammer Generating Station,2480,1,<NA>,2019-01-01,2,0.0,0.0,NaN,...,Tangentially-fired,<NA>,<NA>,<NA>,<NA>,"ARP, CSNOX, CSOSG2, CSSO2G1, RGGI",2019,72.0,0.0,0.0
3,NY,Danskammer Generating Station,2480,1,<NA>,2019-01-01,3,0.0,0.0,NaN,...,Tangentially-fired,<NA>,<NA>,<NA>,<NA>,"ARP, CSNOX, CSOSG2, CSSO2G1, RGGI",2019,72.0,0.0,0.0
4,NY,Danskammer Generating Station,2480,1,<NA>,2019-01-01,4,0.0,0.0,NaN,...,Tangentially-fired,<NA>,<NA>,<NA>,<NA>,"ARP, CSNOX, CSOSG2, CSSO2G1, RGGI",2019,72.0,0.0,0.0
5,NY,Danskammer Generating Station,2480,1,<NA>,2019-01-01,5,0.0,0.0,NaN,...,Tangentially-fired,<NA>,<NA>,<NA>,<NA>,"ARP, CSNOX, CSOSG2, CSSO2G1, RGGI",2019,72.0,0.0,0.0
6,NY,Danskammer Generating Station,2480,1,<NA>,2019-01-01,6,0.0,0.0,NaN,...,Tangentially-fired,<NA>,<NA>,<NA>,<NA>,"ARP, CSNOX, CSOSG2, CSSO2G1, RGGI",2019,72.0,0.0,0.0
7,NY,Danskammer Generating Station,2480,1,<NA>,2019-01-01,7,0.0,0.0,NaN,...,Tangentially-fired,<NA>,<NA>,<NA>,<NA>,"ARP, CSNOX, CSOSG2, CSSO2G1, RGGI",2019,72.0,0.0,0.0
8,NY,Danskammer Generating Station,2480,1,<NA>,2019-01-01,8,0.0,0.0,NaN,...,Tangentially-fired,<NA>,<NA>,<NA>,<NA>,"ARP, CSNOX, CSOSG2, CSSO2G1, RGGI",2019,72.0,0.0,0.0
9,NY,Danskammer Generating Station,2480,1,<NA>,2019-01-01,9,0.0,0.0,NaN,...,Tangentially-fired,<NA>,<NA>,<NA>,<NA>,"ARP, CSNOX, CSOSG2, CSSO2G1, RGGI",2019,72.0,0.0,0.0


,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),Steam Load (1000 lb/hr),SO2 Mass (lbs),...,Winter Capacity (MW),Minimum Load (MW),Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean,Month,Day,hourly_capacity_factor,hourly_ramping_factor
0,NY,Danskammer Generating Station,2480,1,2019-06-04,11,0.08,0.0,NaN,0.000,...,68.2,20.0,<NA>,68,68,68.0,6,6,0.0,0.0
1,NY,Danskammer Generating Station,2480,1,2019-06-10,20,0.72,0.0,NaN,0.072,...,68.2,20.0,<NA>,68,68,68.0,6,6,0.0,0.0
2,NY,Danskammer Generating Station,2480,1,2019-06-10,21,1.00,0.0,NaN,0.100,...,68.2,20.0,<NA>,68,68,68.0,6,6,0.0,0.0
3,NY,Danskammer Generating Station,2480,1,2019-06-10,22,1.00,0.0,NaN,0.100,...,68.2,20.0,<NA>,68,68,68.0,6,6,0.0,0.0
4,NY,Danskammer Generating Station,2480,1,2019-06-10,23,1.00,0.0,NaN,0.100,...,68.2,20.0,<NA>,68,68,68.0,6,6,0.0,0.0


In [ ]:
complete_campd_ny.to_csv('NY/Hourly/Concatted File/ny_2019_2024_complete_cleaned_Operating.csv', index=False)

<h3>Texas</h3>

In [ ]:
complete_campd_tx = pd.read_csv(
    'TX/Hourly/Concatted File/tx_2019_2024_complete_Operating.csv',  
    usecols=complete_campd_columns,
    dtype=complete_campd_type_dict,
    parse_dates=["Date"]
)

# extract day and month as new col from date
complete_campd_tx["Month"] = complete_campd_tx["Date"].dt.month
complete_campd_tx["Day"] = complete_campd_tx["Date"].dt.day

# create full datetime column (date + hour)
campd_tx["datetime"] = pd.to_datetime({
    'year': campd_tx["Date"].dt.year,
    'month': campd_tx["Date"].dt.month,
    'day': campd_tx["Date"].dt.day,
    'hour': campd_tx["Hour"]
})

In [71]:
print(complete_campd_tx.shape)
null_tx = complete_campd_tx.isnull().sum()
null_tx = null_tx[null_tx > 0].sort_values(ascending=False)
display(null_tx)

# Give null vals as percentage of total rows
total_rows = complete_campd_tx.shape[0]
null_percentages = (null_tx / total_rows) * 100

null_percentages.sort_values(ascending=False)

(9262325, 49)


Steam Load (1000 lb/hr)                             9166954
SO2 Controls                                        8476610
NOx Phase                                           8336382
Planned Retirement Year                             8027018
Secondary Fuel Type                                 6761944
Minimum Load (MW)                                   2065624
Winter Capacity (MW)                                2065624
Summer Capacity (MW)                                2065624
Nameplate Capacity (MW) - EIA                       2065624
unit_age_mean                                       1156727
unit_age_max                                        1156727
unit_age_min                                        1156727
NOx Controls                                         332251
Gross Load (MW)                                       95371
Longitude                                             50188
Latitude                                              50188
Operating Status                        

Steam Load (1000 lb/hr)                             98.970334
SO2 Controls                                        91.517087
NOx Phase                                           90.003126
Planned Retirement Year                             86.663100
Secondary Fuel Type                                 73.004823
Minimum Load (MW)                                   22.301355
Winter Capacity (MW)                                22.301355
Summer Capacity (MW)                                22.301355
Nameplate Capacity (MW) - EIA                       22.301355
unit_age_min                                        12.488517
unit_age_mean                                       12.488517
unit_age_max                                        12.488517
NOx Controls                                         3.587123
Gross Load (MW)                                      1.029666
Associated Generators & Nameplate Capacity (MWe)     0.541851
SO2 Phase                                            0.541851
Source C

In [72]:
# handle categorical null values appropriately

cat_cols_with_na_none = [
    "SO2 Controls",
    "NOx Phase",
    "Secondary Fuel Type",
    "NOx Controls",
    "SO2 Phase"
]

cat_cols_with_na_unknown = [
    "Operating Status",
    "Associated Generators & Nameplate Capacity (MWe)",
    "Source Category",
    "CO2 Rate Measure Indicator",
    "SO2 Rate Measure Indicator",
    "NOx Rate Measure Indicator", "Program Code"
]

# here nan means that no data was likely purposely reported -> fill with "Nothing"
for col in cat_cols_with_na_none:
    complete_campd_tx[col] = complete_campd_tx[col].fillna("Nothing").astype("string")

# here nan means that no data was likely reported -> fill with "Missing"
for col in cat_cols_with_na_unknown:
    complete_campd_tx[col] = complete_campd_tx[col].fillna("Missing").astype("string")

num_cols_with_na = [
    "Planned Retirement Year",
    "unit_age_max",
    "unit_age_min",
    "unit_age_mean",
    "Minimum Load (MW)",
    "Winter Capacity (MW)",
    "Summer Capacity (MW)",
    #"Nameplate Capacity (MW) - EIA", # dropped later
    "Longitude",
    "Latitude",
    "Max Hourly HI Rate (mmBtu/hr)",
    "Nameplate Capacity (MWe)"
]

# we drop nameplate capacity (MW) - EIA (redundant with Nameplate Capacity (MWe) and has many nulls)
complete_campd_tx = complete_campd_tx.drop(columns=["Nameplate Capacity (MW) - EIA"])

print("Null value percentages in complete_campd_tx after adjustments:")
total_rows = complete_campd_tx.shape[0]
null_tx = complete_campd_tx.isnull().sum()
null_tx = null_tx[null_tx > 0].sort_values(ascending=False)
null_percentages = (null_tx / total_rows) * 100
null_percentages.sort_values(ascending=False)

Null value percentages in complete_campd_tx after adjustments:


Steam Load (1000 lb/hr)          98.970334
Planned Retirement Year          86.663100
Summer Capacity (MW)             22.301355
Winter Capacity (MW)             22.301355
Minimum Load (MW)                22.301355
unit_age_min                     12.488517
unit_age_max                     12.488517
unit_age_mean                    12.488517
Gross Load (MW)                   1.029666
Latitude                          0.541851
Longitude                         0.541851
Max Hourly HI Rate (mmBtu/hr)     0.541851
Nameplate Capacity (MWe)          0.541851
SO2 Rate (lbs/mmBtu)              0.014586
CO2 Rate (short tons/mmBtu)       0.014586
NOx Rate (lbs/mmBtu)              0.014327
dtype: float64

In [73]:
# measurement methods
print("Measurement methods in complete_campd_tx:")
display(complete_campd_tx.groupby("CO2 Mass Measure Indicator")["CO2 Mass (short tons)"].count())
display(complete_campd_tx.groupby("NOx Mass Measure Indicator")["NOx Mass Measure Indicator"].count())
display(complete_campd_tx.groupby("SO2 Mass Measure Indicator")["SO2 Mass Measure Indicator"].count())

Measurement methods in complete_campd_tx:


CO2 Mass Measure Indicator
LME                          43060
Measured                   8767620
Measured and Substitute     352561
Other                        87856
Substitute                   11228
Name: CO2 Mass (short tons), dtype: int64

NOx Mass Measure Indicator
Calculated                  359450
LME                          43060
Measured                   8445571
Measured and Substitute     412778
Substitute                    1466
Name: NOx Mass Measure Indicator, dtype: int64

SO2 Mass Measure Indicator
LME                          43060
Measured                   8914828
Measured and Substitute      67393
Other                         2779
Substitute                  234265
Name: SO2 Mass Measure Indicator, dtype: int64

In [74]:
# calculate hourly capacity factor
complete_campd_tx["hourly_capacity_factor"] = (
    complete_campd_tx["Gross Load (MW)"] * complete_campd_tx["Operating Time"]
) / complete_campd_tx["Nameplate Capacity (MWe)"]

complete_campd_tx.head(5)

,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),Steam Load (1000 lb/hr),SO2 Mass (lbs),...,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean,Month,Day,hourly_capacity_factor
0,TX,Oklaunion Power Station,127,1,2019-01-01,0,1.0,219.0,NaN,280.9,...,NaN,NaN,NaN,2020,33,33,33.0,1,1,0.329869
1,TX,Oklaunion Power Station,127,1,2019-01-01,1,1.0,221.0,NaN,286.0,...,NaN,NaN,NaN,2020,33,33,33.0,1,1,0.332881
2,TX,Oklaunion Power Station,127,1,2019-01-01,2,1.0,223.0,NaN,289.2,...,NaN,NaN,NaN,2020,33,33,33.0,1,1,0.335894
3,TX,Oklaunion Power Station,127,1,2019-01-01,3,1.0,223.0,NaN,293.2,...,NaN,NaN,NaN,2020,33,33,33.0,1,1,0.335894
4,TX,Oklaunion Power Station,127,1,2019-01-01,4,1.0,222.0,NaN,293.0,...,NaN,NaN,NaN,2020,33,33,33.0,1,1,0.334388


In [ ]:
# calculate hourly ramping factor
filtered_out_rows_campd_tx = pd.read_csv('TX/Hourly/Concatted File/tx_2019_2024_NonOperating.csv', dtype=campd_type_dict, parse_dates=["Date"])
filtered_out_rows_campd_tx["Year"] = filtered_out_rows_campd_tx["Date"].dt.year
facility_small = pd.read_csv('Facility/Concatted File/facility_2014_2024.csv', dtype={"Facility ID": "Int64", "Unit ID": "string", "Year": "Int64"})
facility_small = facility_small[["Year", 'Facility ID', 'Unit ID', "Nameplate Capacity (MWe)"]]
all_hours_tx = filtered_out_rows_campd_tx.merge(facility_small, on=['Year', 'Facility ID', 'Unit ID'], how='left')
all_hours_tx["Gross Load (MW)"] = all_hours_tx["Gross Load (MW)"].fillna(0)

all_hours_tx = pd.concat([all_hours_tx, complete_campd_tx[["Facility ID", "Unit ID", "Date", "Hour", "Gross Load (MW)", "Year", "Nameplate Capacity (MWe)"]]], ignore_index=True)

# Sort data by unit and time to calculate t-1 values
all_hours_tx = all_hours_tx.sort_values(["Facility ID", "Unit ID", "Date", "Hour"])

# Calculate lagged gross load per unit (get previous row's value for gross load)
all_hours_tx["gross_load_lag1"] = all_hours_tx.groupby(["Facility ID", "Unit ID"])["Gross Load (MW)"].shift(1)

# calc ramping factor
all_hours_tx["hourly_ramping_factor"] = (
    all_hours_tx["Gross Load (MW)"] - all_hours_tx["gross_load_lag1"]
) / all_hours_tx["Nameplate Capacity (MWe)"]

display(all_hours_tx.head(10))

complete_campd_tx = complete_campd_tx.merge(all_hours_tx[["Facility ID", "Unit ID", "Date", "Hour", "hourly_ramping_factor"]],
                        on=["Facility ID", "Unit ID", "Date", "Hour"],
                        how="left")
complete_campd_tx.head(5)

,State,Facility Name,Facility ID,Unit ID,Associated Stacks,Date,Hour,Operating Time,Gross Load (MW),Steam Load (1000 lb/hr),...,Unit Type,SO2 Controls,NOx Controls,PM Controls,Hg Controls,Program Code,Year,Nameplate Capacity (MWe),gross_load_lag1,hourly_ramping_factor
0,TX,Copper Station,9,CTG-1,<NA>,2019-01-01,0,0.0,0.0,NaN,...,Combustion turbine,<NA>,Water Injection,<NA>,<NA>,CSOSG2,2019,NaN,NaN,NaN
1,TX,Copper Station,9,CTG-1,<NA>,2019-01-01,1,0.0,0.0,NaN,...,Combustion turbine,<NA>,Water Injection,<NA>,<NA>,CSOSG2,2019,NaN,0.0,NaN
2,TX,Copper Station,9,CTG-1,<NA>,2019-01-01,2,0.0,0.0,NaN,...,Combustion turbine,<NA>,Water Injection,<NA>,<NA>,CSOSG2,2019,NaN,0.0,NaN
3,TX,Copper Station,9,CTG-1,<NA>,2019-01-01,3,0.0,0.0,NaN,...,Combustion turbine,<NA>,Water Injection,<NA>,<NA>,CSOSG2,2019,NaN,0.0,NaN
4,TX,Copper Station,9,CTG-1,<NA>,2019-01-01,4,0.0,0.0,NaN,...,Combustion turbine,<NA>,Water Injection,<NA>,<NA>,CSOSG2,2019,NaN,0.0,NaN
5,TX,Copper Station,9,CTG-1,<NA>,2019-01-01,5,0.0,0.0,NaN,...,Combustion turbine,<NA>,Water Injection,<NA>,<NA>,CSOSG2,2019,NaN,0.0,NaN
6,TX,Copper Station,9,CTG-1,<NA>,2019-01-01,6,0.0,0.0,NaN,...,Combustion turbine,<NA>,Water Injection,<NA>,<NA>,CSOSG2,2019,NaN,0.0,NaN
7,TX,Copper Station,9,CTG-1,<NA>,2019-01-01,7,0.0,0.0,NaN,...,Combustion turbine,<NA>,Water Injection,<NA>,<NA>,CSOSG2,2019,NaN,0.0,NaN
8,TX,Copper Station,9,CTG-1,<NA>,2019-01-01,18,0.0,0.0,NaN,...,Combustion turbine,<NA>,Water Injection,<NA>,<NA>,CSOSG2,2019,NaN,0.0,NaN
9,TX,Copper Station,9,CTG-1,<NA>,2019-01-01,19,0.0,0.0,NaN,...,Combustion turbine,<NA>,Water Injection,<NA>,<NA>,CSOSG2,2019,NaN,0.0,NaN


,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),Steam Load (1000 lb/hr),SO2 Mass (lbs),...,Winter Capacity (MW),Minimum Load (MW),Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean,Month,Day,hourly_capacity_factor,hourly_ramping_factor
0,TX,Oklaunion Power Station,127,1,2019-01-01,0,1.0,219.0,NaN,280.9,...,NaN,NaN,2020,33,33,33.0,1,1,0.329869,NaN
1,TX,Oklaunion Power Station,127,1,2019-01-01,1,1.0,221.0,NaN,286.0,...,NaN,NaN,2020,33,33,33.0,1,1,0.332881,0.003013
2,TX,Oklaunion Power Station,127,1,2019-01-01,2,1.0,223.0,NaN,289.2,...,NaN,NaN,2020,33,33,33.0,1,1,0.335894,0.003013
3,TX,Oklaunion Power Station,127,1,2019-01-01,3,1.0,223.0,NaN,293.2,...,NaN,NaN,2020,33,33,33.0,1,1,0.335894,0.000000
4,TX,Oklaunion Power Station,127,1,2019-01-01,4,1.0,222.0,NaN,293.0,...,NaN,NaN,2020,33,33,33.0,1,1,0.334388,-0.001506


In [ ]:
complete_campd_tx.to_csv('TX/Hourly/Concatted File/tx_2019_2024_complete_cleaned_Operating.csv', index=False)